# 서버 점검 — 디스크·쿼터·주피터 DB (저장 안 될 때)

`attempt to write a readonly database` = 주피터가 쓸 공간이 없거나 db 가 잠긴 것.
① 홈 쓰기 가능한가 → ② 디스크/쿼터 → ③ 뭐가 큰가 → ④ jupyter db → ⑤ 정리(로그 삭제·db 초기화).

저장이 안 돼도 **셀 실행은 됩니다**. (이 노트북은 git 에서 받은 것)


In [ ]:
import os, subprocess, shutil
from pathlib import Path

HOME = Path.home()
# 출력 경로: 환경변수 우선, 없으면 기본
OUT = Path(os.environ.get('LEROBOT_OUTPUT', str(HOME / 'lerobot_project/outputs/final')))
LOGS = OUT / '_logs'

def sh(cmd, limit=40):
    """셸 명령 실행 후 출력(최대 limit 줄) 표시. 명령 없으면 조용히 넘어감."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=120)
    except Exception as e:
        print(f'  (실행 실패: {e})'); return
    out = (r.stdout or '') + (r.stderr or '')
    lines = [l for l in out.splitlines() if l.strip()]
    for l in lines[:limit]:
        print(l)
    if len(lines) > limit:
        print(f'  ... (+{len(lines) - limit}줄)')
    if not lines:
        print('  (출력 없음)')

print('HOME:', HOME)
print('OUT :', OUT, '| 존재:', OUT.is_dir())
print('USER:', os.environ.get('USER', '?'))

## ① 홈 쓰기 테스트 (어디가 막혔나)


In [ ]:
# ── ① 홈이 진짜 쓰기 가능한가? (readonly database = 여기가 막힌 것) ──
for label, d in [('HOME', HOME), ('.jupyter', HOME / '.jupyter'),
                 ('.local/share/jupyter', HOME / '.local/share/jupyter'), ('OUT', OUT)]:
    try:
        d.mkdir(parents=True, exist_ok=True)
        tf = d / '.__writetest'
        tf.write_text('x'); tf.unlink()
        print(f'  ✅ {label:<22} 쓰기 OK  ({d})')
    except Exception as e:
        print(f'  ❌ {label:<22} 쓰기 실패: {type(e).__name__} {e}')
print('\n→ ❌ 뜨는 곳이 꽉 찼거나 readonly. 아래 ②③ 로 원인(용량/쿼터) 확인.')

## ② 디스크 / 쿼터


In [ ]:
# ── ② 디스크 사용량 / 쿼터 ──
print('=== df -h (HOME · OUT 가 있는 FS) ===')
sh(f'df -h {HOME} {OUT} 2>/dev/null')
print('\n=== 쿼터 (있으면) ===')
sh('lfs quota -h -u $USER /home1 2>/dev/null')
sh('quota -s 2>/dev/null')
print('\n※ Use% 가 100% 거나 쿼터 초과면 그게 원인.')

## ③ 뭐가 공간을 먹나


In [ ]:
# ── ③ 뭐가 공간을 먹나 — OUT 하위 + 로그 총량 ──
print('=== OUT 하위 디렉토리 크기 ===')
sh(f'du -sh {OUT}/* 2>/dev/null | sort -h')
print(f'\n=== 로그 총량 ({LOGS}) ===')
sh(f'du -sh {LOGS} 2>/dev/null')
sh(f'ls -1 {LOGS}/*.log 2>/dev/null | wc -l')
print('  ↑ .log 개수')
print('\n=== 큰 파일 top 15 (OUT 아래) ===')
sh(f'du -ah {OUT} 2>/dev/null | sort -h | tail -15')

## ④ 주피터 SQLite DB


In [ ]:
# ── ④ 주피터 SQLite DB 위치·권한 (readonly 로 걸린 것) ──
print('=== jupyter/ystore/file_id db 찾기 ===')
sh(f'find {HOME} -maxdepth 5 -name "*.db" 2>/dev/null | grep -iE "jupyter|ystore|file_id"')
sh(f'find {OUT.parent} -maxdepth 4 -name ".jupyter_ystore.db*" -o -name "file_id_manager.db*" 2>/dev/null')
print('\n=== 권한/크기 ===')
sh(f'ls -la {HOME}/.local/share/jupyter/*.db 2>/dev/null')
sh('ls -la ~/.jupyter_ystore.db* 2>/dev/null')

## ⑤ 정리 — 로그 삭제 + db 초기화 (dry-run → EXECUTE=True)
**train/ · eval_clean/(실제 결과)는 안 지움.** 로그·db 만. db 지운 뒤 jupyter 재시작.


In [ ]:
# ── ⑤ 정리 (공간 확보 + 잠긴 db 초기화) ── 확인 후 EXECUTE=True ──
#    · 로그(.log): 그냥 stdout 캡처라 지워도 결과 데이터 아님(안전).
#    · jupyter db: 지우면 서버가 다시 만든다. **jupyter 저장 문제 해결.**
#    ⚠️ 실제 결과(train/ · eval_clean/)는 안 건드림.
EXECUTE = False

# (a) 로그 삭제
logs = list(LOGS.glob('*.log')) if LOGS.is_dir() else []
tot = sum(p.stat().st_size for p in logs) / 1e9
print(f'{"삭제" if EXECUTE else "DRY-RUN"} — 로그 {len(logs)}개 ({tot:.1f} GB)')
if EXECUTE:
    for p in logs:
        try:
            p.unlink()
        except Exception as e:
            print('  실패:', p.name, e)
    print('  로그 삭제 완료')

# (b) 잠긴 jupyter db 초기화 (파일만; 서버가 재생성)
cands = list((HOME / '.local/share/jupyter').glob('file_id_manager.db*'))
cands += list(HOME.glob('.jupyter_ystore.db*'))
print(f'\n{"삭제" if EXECUTE else "DRY-RUN"} — jupyter db {len(cands)}개:')
for p in cands:
    print('   ', p)
    if EXECUTE:
        try:
            p.unlink()
        except Exception as e:
            print('     실패:', e)
print('\n' + ('완료 → jupyter 서버 재시작 후 저장 확인.' if EXECUTE
               else '확인됐으면 EXECUTE=True. db 지운 뒤엔 jupyter 서버 재시작 필요.'))